# Minimum acquisition for NODDI + soma (Tort2) at SNR=20

## Aim

Use repeated noisy simulations and least-squares fitting to answer:

> **What is the minimum acquisition needed to fit the full NODDI+soma model reliably?**

The model used throughout is:

- neurite + extra-neurite + soma + free-water compartments
- directional data
- **Tort2 soma-aware tortuosity approximation**
- `d_in`, `d_soma`, and `d_fw`/`d_csf` fixed in the fit
- `theta` and `phi` fixed using DTI-informed orientation
- moderate NODDI-informed Gaussian-style priors on `kappa`, `f_neurite`, `f_extra`, and `f_fw`
- `r_soma` and `f_soma` are the main biological parameters of interest
- `S0` remains free

The notebook has three stages:

1. **LS acquisition comparison** across a user-supplied list of acquisitions.
2. **MCMC on the best 3 acquisitions** to examine posterior correlations/degeneracies.
3. **Fixed-diffusivity robustness**: deliberately fix incorrect `d_in`, `d_soma`, or `d_fw` values and quantify bias in the recovered soma parameters.

The structure follows the previous Stage 1 notebook: synthetic truth scenarios, prescribed NODDI errors, simplex-constrained fractions, checkpointed repeated LS fits, recovery metrics, and saved figures. fileciteturn8file0L3-L32

In [1]:
from pathlib import Path
from dataclasses import dataclass
import sys
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")


repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dmri_models import (
    Ball,
    Component,
    Model,
    ModelPreset,
    Parameter,
    ParameterSpace,
    Simplex,
    Sphere,
    Stick,
)
from dmri_models.acquisitions import rm_full, rm_10_shells_bmax_5, rm_no_b0p5, rm_no_b5, rm_no_b4, rm_no_b4_or_b5, rm_no_b3_or_b4, rm_no_b2_or_b4, rm_no_b3_b4_or_b5
from dmri_models.compartments import WatsonFOD, FODConvolved


In [ ]:
# =============================================================================
# 1. USER CONFIGURATION
# =============================================================================

SNR = 20.0

# Initial LS benchmark. Increase for the final run.
N_REPEATS = 1
BASE_SEED = 7000
N_STARTS = 1
MAX_NFEV = 10

# Which truth scenarios to benchmark.
TRUTH_SCENARIOS_TO_RUN = ["truth_1", "truth_2", "truth_3"]

ACQUISITIONS = {
    "rm_full": rm_full(),
    # "rm_10_shells_bmax_5": rm_10_shells_bmax_5(),
    # "rm_no_b0p5": rm_no_b0p5(),
    # "rm_no_b5": rm_no_b5(),
    # "rm_no_b4": rm_no_b4(),
    # "rm_no_b4_or_b5": rm_no_b4_or_b5(),
    # "rm_no_b3_or_b4": rm_no_b3_or_b4(),
    # "rm_no_b2_or_b4": rm_no_b2_or_b4(),
    # "rm_no_b3_b4_or_b5": rm_no_b3_b4_or_b5(),
}

# -----------------------------------------------------------------------------
# DTI-informed orientation.
# Supply a per-acquisition (theta, phi) if DTI gives different estimates for
# each acquisition. Defaults here are the truth orientation used in the
# synthetic simulations.
# -----------------------------------------------------------------------------

DEFAULT_THETA = 1.0
DEFAULT_PHI = 1.2
ORIENTATION_BY_ACQUISITION = {
    name: (DEFAULT_THETA, DEFAULT_PHI)
    for name in ACQUISITIONS
}

# -----------------------------------------------------------------------------
# Operational definition of a "passing" acquisition. These are deliberately
# configurable rather than treated as universal requirements.
# -----------------------------------------------------------------------------

MAX_MEAN_PRIMARY_REL_RMSE = 0.20
MAX_WORST_SCENARIO_PRIMARY_REL_RMSE = 0.30
MIN_SUCCESS_RATE = 0.95

# -----------------------------------------------------------------------------
# MCMC settings for the best 3 acquisitions.
# -----------------------------------------------------------------------------

MCMC_TRUTH_SCENARIO = "truth_2"
MCMC_CHAINS = 4
MCMC_STEPS = 3000
MCMC_BURN = 1000
MCMC_THIN = 5
MCMC_SEED = 9100

# Proposal scales in optimisation coordinates:
# [S0, r_soma, kappa, z_neurite, z_extra, z_fw]
MCMC_PROPOSAL_SCALES = np.array([
    0.01,
    0.12,
    0.08,
    0.08,
    0.08,
    0.08,
])

# -----------------------------------------------------------------------------
# Fixed-diffusivity robustness. The fit always fixes the model to the baseline
# values below, while the synthetic truth is deliberately moved away from them.
# -----------------------------------------------------------------------------

FIXED_MODEL_DIFFUSIVITIES = {
    "d_in": 2.2,
    "d_soma": 3.0,
    "d_csf": 3.0,
}

MISFIX_FACTORS = {
    "d_in": np.array([0.60, 0.80, 1.00, 1.20]),
    "d_soma": np.array([0.60, 0.80, 1.00, 1.20]),
    "d_csf": np.array([0.60, 0.80, 1.00, 1.20]),
}

ROBUSTNESS_REPEATS = 20
ROBUSTNESS_MODE = "one_at_a_time"  # "one_at_a_time" or "full_grid"

LS_RESULTS_FILE = Path("minimum_acquisition_tort2_ls_checkpoint.csv")
ROBUSTNESS_RESULTS_FILE = Path("minimum_acquisition_tort2_fixed_diffusivity_checkpoint.csv")
MCMC_RESULTS_DIR = Path("minimum_acquisition_tort2_mcmc")
PLOT_DIR = Path("minimum_acquisition_tort2_plots")
MCMC_RESULTS_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)


In [ ]:
# =============================================================================
# 2. TRUTH SCENARIOS + NODDI PERFORMANCE MODEL
# =============================================================================

TRUE_BASE = {
    "S0": 1.0,
    "d_in": 2.2,
    "d_soma": 3.0,
    "d_csf": 3.0,
    "kappa": 2.0,
    "theta": DEFAULT_THETA,
    "phi": DEFAULT_PHI,
}

TRUTH_SCENARIOS = {
    "truth_1": {
        "label": "f_soma=0.10, r_soma=5, f_fw=0.15",
        "soma_fraction": 0.10,
        "r_soma": 5.0,
        "csf_fraction": 0.15,
    },
    "truth_2": {
        "label": "f_soma=0.15, r_soma=7, f_fw=0.20",
        "soma_fraction": 0.15,
        "r_soma": 7.0,
        "csf_fraction": 0.20,
    },
    "truth_3": {
        "label": "f_soma=0.20, r_soma=8, f_fw=0.25",
        "soma_fraction": 0.20,
        "r_soma": 8.0,
        "csf_fraction": 0.25,
    },
}


def make_truth(scenario, neurite_to_extra=2.0):
    f_soma = float(scenario["soma_fraction"])
    f_fw = float(scenario["csf_fraction"])
    remaining = 1.0 - f_soma - f_fw
    f_extra = remaining / (1.0 + neurite_to_extra)
    f_neurite = neurite_to_extra * f_extra

    truth = dict(TRUE_BASE)
    truth.update({
        "r_soma": float(scenario["r_soma"]),
        "neurite_fraction": f_neurite,
        "extra_fraction": f_extra,
        "soma_fraction": f_soma,
        "csf_fraction": f_fw,
    })
    return truth


TRUTH_BY_SCENARIO = {
    name: make_truth(scenario)
    for name, scenario in TRUTH_SCENARIOS.items()
}

# Reuse the conservative NODDI uncertainty assumptions from the previous
# notebook, unless a user supplies acquisition-specific values below.
DEFAULT_NODDI_MAE = {
    "kappa": 0.15,
    "extra_fraction": 0.06,
    "neurite_fraction": 0.02,
    "csf_fraction": 0.04,
}

# Optional acquisition-specific NODDI MAE overrides.
NODDI_MAE_BY_ACQUISITION = {
    # "rm_full": {"kappa": 0.15, "extra_fraction": 0.06, "neurite_fraction": 0.02, "csf_fraction": 0.04,},
    # "rm_10_shells_bmax_5": {"kappa": 0.15, "extra_fraction": 0.06, "neurite_fraction": 0.02, "csf_fraction": 0.04,},
    # "rm_no_b0p5": {"kappa": 0.15, "extra_fraction": 0.075, "neurite_fraction": 0.02, "csf_fraction": 0.06,},
    # "rm_no_b5": {"kappa": 0.15, "extra_fraction": 0.06, "neurite_fraction": 0.02, "csf_fraction": 0.04,},
    # "rm_no_b4": {"kappa": 0.15, "extra_fraction": 0.06, "neurite_fraction": 0.02, "csf_fraction": 0.04,},
    # "rm_no_b4_or_b5": {"kappa": 0.15, "extra_fraction": 0.075, "neurite_fraction": 0.025, "csf_fraction": 0.06,},
    # "rm_no_b3_or_b4": {"kappa": 0.15, "extra_fraction": 0.075, "neurite_fraction": 0.025, "csf_fraction": 0.06,},
    # "rm_no_b2_or_b4": {"kappa": 0.15, "extra_fraction": 0.075, "neurite_fraction": 0.025, "csf_fraction": 0.06,},
    # "rm_no_b3_b4_or_b5": {"kappa": 0.20, "extra_fraction": 0.075, "neurite_fraction": 0.025, "csf_fraction": 0.06,},
}

NODDI_ESTIMATE_MODE = "per_repeat"  # or "fixed_per_truth"
NODDI_SEED = 24680


def noddi_mae(acquisition_name):
    out = dict(DEFAULT_NODDI_MAE)
    out.update(NODDI_MAE_BY_ACQUISITION.get(acquisition_name, {}))
    return out


def make_noddi_estimate(truth, acquisition_name, seed):
    rng = np.random.default_rng(seed)
    mae = noddi_mae(acquisition_name)

    estimate = {
        "kappa": max(0.0, truth["kappa"] + rng.laplace(0.0, mae["kappa"])),
        "neurite_fraction": max(0.0, truth["neurite_fraction"] + rng.laplace(0.0, mae["neurite_fraction"])),
        "extra_fraction": max(0.0, truth["extra_fraction"] + rng.laplace(0.0, mae["extra_fraction"])),
        "csf_fraction": max(0.0, truth["csf_fraction"] + rng.laplace(0.0, mae["csf_fraction"])),
    }

    nuisance = np.array([
        estimate["neurite_fraction"],
        estimate["extra_fraction"],
        estimate["csf_fraction"],
    ])
    if nuisance.sum() >= 0.98:
        nuisance *= 0.98 / nuisance.sum()

    estimate["neurite_fraction"] = float(nuisance[0])
    estimate["extra_fraction"] = float(nuisance[1])
    estimate["csf_fraction"] = float(nuisance[2])
    return estimate


truth_df = pd.DataFrame([
    {
        "scenario": name,
        "r_soma": truth["r_soma"],
        "f_soma": truth["soma_fraction"],
        "f_fw": truth["csf_fraction"],
        "f_neurite": truth["neurite_fraction"],
        "f_extra": truth["extra_fraction"],
    }
    for name, truth in TRUTH_BY_SCENARIO.items()
])

display(truth_df)


In [ ]:
# =============================================================================
# 3. TORT2 MODEL
# =============================================================================


def _nonnegative_parameter(parameters, name):
    value = float(parameters[name])
    if not np.isfinite(value) or value < 0:
        raise ValueError(f"{name} must be finite and non-negative")
    return value


@dataclass
class NODDISomaTortuosityExtraNeurite:
    """Extra-neurite Zeppelin using the soma-aware Tort2 approximation.

    d_ex_parallel = d_in * (1 - f_soma / (1 - f_csf))

    d_ex_perp = d_ex_parallel * (1 - f_neurite / (1 - f_csf))
    """

    parallel_diffusivity: str
    orientation: str
    neurite_fraction: str = "neurite_fraction"
    soma_fraction: str = "soma_fraction"
    csf_fraction: str | None = "csf_fraction"

    def signal(self, acquisition, parameters):
        if acquisition.gradients is None:
            raise ValueError("A tensor requires acquisition gradient directions.")

        d_in = _nonnegative_parameter(
            parameters,
            self.parallel_diffusivity,
        )

        f_neurite = float(parameters[self.neurite_fraction])
        f_soma = float(parameters[self.soma_fraction])

        if not np.isfinite(f_neurite) or f_neurite < 0:
            raise ValueError("neurite fraction must be finite and non-negative")
        if not np.isfinite(f_soma) or f_soma < 0:
            raise ValueError("soma fraction must be finite and non-negative")

        if self.csf_fraction is None:
            f_csf = 0.0
        else:
            f_csf = float(parameters[self.csf_fraction])
            if not np.isfinite(f_csf) or f_csf < 0 or f_csf >= 1.0:
                raise ValueError("csf fraction must be finite, non-negative and < 1")

        available = max(1e-12, 1.0 - f_csf)
        soma_factor = 1.0 - f_soma / available
        v_ic = f_neurite / available

        if soma_factor < 0:
            raise ValueError("Soma factor would produce negative d_ex_parallel")
        if not 0 <= v_ic <= 1:
            raise ValueError("Computed intracellular fraction must lie in [0, 1]")

        d_parallel = d_in * soma_factor
        d_perpendicular = d_parallel * (1.0 - v_ic)

        orientation = np.asarray(
            parameters[self.orientation],
            dtype=float,
        ).reshape(-1)
        if orientation.shape != (3,) or not np.all(np.isfinite(orientation)):
            raise ValueError(f"{self.orientation} must be a finite three-vector")

        norm = np.linalg.norm(orientation)
        if np.isclose(norm, 0.0):
            raise ValueError(f"{self.orientation} must be non-zero")
        orientation = orientation / norm

        projection = acquisition.gradients @ orientation
        beta = (
            np.ones(acquisition.n_measurements)
            if acquisition.b_tensor_shape is None
            else acquisition.b_tensor_shape
        )

        directional_factor = (
            beta * (projection ** 2 - 1.0 / 3.0)
            + 1.0 / 3.0
        )

        adc_term = (
            (d_parallel - d_perpendicular)
            * directional_factor
            + d_perpendicular
        )

        return np.exp(-acquisition.b_values * adc_term)


def tort2_model(
    *,
    theta=DEFAULT_THETA,
    phi=DEFAULT_PHI,
):
    fod = WatsonFOD(
        kappa="kappa",
        theta="theta",
        phi="phi",
        n_theta=20,
        n_phi=40,
    )

    model = Model([
        Component(
            "neurite",
            compartment=FODConvolved(
                response=Stick(
                    diffusivity="d_in",
                    orientation="axis",
                ),
                fod=fod,
                orientation_parameter="axis",
            ),
        ),
        Component(
            "extra",
            compartment=FODConvolved(
                response=NODDISomaTortuosityExtraNeurite(
                    parallel_diffusivity="d_in",
                    orientation="axis",
                    neurite_fraction="neurite_fraction",
                    soma_fraction="soma_fraction",
                    csf_fraction="csf_fraction",
                ),
                fod=fod,
                orientation_parameter="axis",
            ),
        ),
        Component(
            "soma",
            compartment=Sphere(
                radius="r_soma",
                diffusivity="d_soma",
            ),
        ),
        Component(
            "csf",
            compartment=Ball(
                diffusivity="d_csf",
            ),
        ),
    ])

    parameters = ParameterSpace([
        Parameter("S0", 1.0, (0.05, 2.0)),
        Parameter("d_in", FIXED_MODEL_DIFFUSIVITIES["d_in"], (0.5, 3.0), vary=False),
        Parameter("d_soma", FIXED_MODEL_DIFFUSIVITIES["d_soma"], (0.5, 3.5), vary=False),
        Parameter("d_csf", FIXED_MODEL_DIFFUSIVITIES["d_csf"], (0.5, 3.5), vary=False),
        Parameter("r_soma", 6.0, (1.0, 15.0), vary=True),
        Parameter("kappa", 2.0, (0.0, 64.0), vary=True),
        Parameter("theta", theta, (0.0, np.pi), vary=False),
        Parameter("phi", phi, (-np.pi, np.pi), vary=False),
    ], simplexes=[
        Simplex(
            ("neurite_fraction", "extra_fraction", "soma_fraction", "csf_fraction"),
            (0.50, 0.25, 0.10, 0.15),
        )
    ])

    return ModelPreset(model=model, parameters=parameters)


In [ ]:
# =============================================================================
# 4. FITTING PARAMETERISATION
# =============================================================================
#
# We use additive log-ratio coordinates rather than four redundant softmax
# logits. This gives exactly three independent fraction coordinates:
#
# z_n = log(f_neurite / f_soma)
# z_e = log(f_extra   / f_soma)
# z_fw = log(f_fw      / f_soma)
#
# This removes the global-shift redundancy that would otherwise be present in
# four unconstrained softmax logits.


def fractions_to_alr(f_neurite, f_extra, f_soma, f_fw):
    fractions = np.array([
        f_neurite,
        f_extra,
        f_fw,
    ], dtype=float)
    f_soma = float(f_soma)
    fractions = np.clip(fractions, 1e-10, None)
    f_soma = max(1e-10, f_soma)
    return np.log(fractions / f_soma)


def alr_to_fractions(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -30.0, 30.0)
    e = np.exp(z)
    denominator = 1.0 + np.sum(e)
    f_soma = 1.0 / denominator
    f_neurite, f_extra, f_fw = e / denominator
    return {
        "neurite_fraction": float(f_neurite),
        "extra_fraction": float(f_extra),
        "soma_fraction": float(f_soma),
        "csf_fraction": float(f_fw),
    }


def alr_jacobian_log(fractions):
    # For additive log-ratio coordinates relative to f_soma, the absolute
    # Jacobian determinant for (f_neurite, f_extra, f_fw) wrt z is the product
    # of all four simplex fractions. This is needed for a mathematically
    # consistent MCMC target when priors are specified on physical fractions.
    vals = np.array([
        fractions["neurite_fraction"],
        fractions["extra_fraction"],
        fractions["soma_fraction"],
        fractions["csf_fraction"],
    ])
    return float(np.sum(np.log(np.clip(vals, 1e-12, None))))


def simulate_signal(preset, acquisition, parameters):
    return np.asarray(
        preset.model.signal(
            acquisition,
            parameters,
        ),
        dtype=float,
    )


def add_gaussian_noise(signal, snr, seed):
    signal = np.asarray(signal, dtype=float)
    rng = np.random.default_rng(seed)
    sigma = float(signal.max()) / float(snr)
    return signal + rng.normal(0.0, sigma, size=signal.shape)


In [ ]:
# =============================================================================
# 5. LS FITTER: TORT2 + MODERATE NODDI-INFORMED GAUSSIAN PENALTIES
# =============================================================================

# Moderate prior sigma = 2 x the conservative NODDI MAE, following the logic
# of the previous notebook. fileciteturn8file0L25-L32

def fit_tort2_ls(
    data,
    acquisition,
    acquisition_name,
    truth,
    noddi,
    *,
    seed=0,
    fixed_diffusivities=None,
):
    rng = np.random.default_rng(seed)
    preset = tort2_model(
        theta=truth["theta"],
        phi=truth["phi"],
    )

    fixed = dict(FIXED_MODEL_DIFFUSIVITIES)
    if fixed_diffusivities is not None:
        fixed.update(fixed_diffusivities)

    mae = noddi_mae(acquisition_name)
    prior_sigma = {
        key: 2.0 * value
        for key, value in mae.items()
    }

    sigma_data = float(
        np.max(np.abs(data)) / SNR
    )

    # Parameter vector: [S0, r_soma, kappa, z_n, z_e, z_fw]
    x0 = np.array([
        1.0,
        7.0,
        noddi["kappa"],
        *fractions_to_alr(
            noddi["neurite_fraction"],
            noddi["extra_fraction"],
            0.15,
            noddi["csf_fraction"],
        ),
    ], dtype=float)

    # Ensure the initial fraction vector is exactly valid by using its
    # physical values and the desired neutral soma starting point.
    nuisance = np.array([
        noddi["neurite_fraction"],
        noddi["extra_fraction"],
        noddi["csf_fraction"],
    ])
    nuisance *= 0.85 / nuisance.sum()
    x0[3:] = fractions_to_alr(
        nuisance[0],
        nuisance[1],
        0.15,
        nuisance[2],
    )

    lower = np.array([
        0.05,
        1.0,
        0.0,
        -20.0,
        -20.0,
        -20.0,
    ])
    upper = np.array([
        2.0,
        15.0,
        64.0,
        20.0,
        20.0,
        20.0,
    ])

    def unpack(x):
        fractions = alr_to_fractions(x[3:])
        return {
            "S0": float(x[0]),
            "r_soma": float(x[1]),
            "kappa": float(x[2]),
            "theta": truth["theta"],
            "phi": truth["phi"],
            **fractions,
            "d_in": float(fixed["d_in"]),
            "d_soma": float(fixed["d_soma"]),
            "d_csf": float(fixed["d_csf"]),
        }

    def residual(x):
        values = unpack(x)
        prediction = simulate_signal(
            preset,
            acquisition,
            values,
        )

        residuals = [
            *((prediction - data) / sigma_data)
        ]

        # Moderate Gaussian-style NODDI priors on all nuisance parameters.
        for parameter in [
            "kappa",
            "neurite_fraction",
            "extra_fraction",
            "csf_fraction",
        ]:
            residuals.append(
                (
                    values[parameter]
                    - noddi[parameter]
                )
                / prior_sigma[parameter]
            )

        return np.asarray(
            residuals,
            dtype=float,
        )

    starts = [x0.copy()]
    for _ in range(max(0, N_STARTS - 1)):
        start = x0.copy()
        start[0] += rng.normal(0.0, 0.03)
        start[1] += rng.normal(0.0, 1.0)
        start[2] = max(0.0, start[2] + rng.normal(0.0, 0.5))
        start[3:] += rng.normal(0.0, 0.20, size=3)
        starts.append(np.clip(start, lower, upper))

    candidates = []
    for start in starts:
        try:
            candidates.append(
                least_squares(
                    residual,
                    start,
                    bounds=(lower, upper),
                    max_nfev=MAX_NFEV,
                )
            )
        except ValueError:
            continue

    if not candidates:
        raise RuntimeError("All LS starts failed")

    best = min(
        candidates,
        key=lambda result: result.cost,
    )
    fitted = unpack(best.x)
    prediction = simulate_signal(
        preset,
        acquisition,
        fitted,
    )

    return {
        **fitted,
        "success": bool(best.success),
        "n_evaluations": int(best.nfev),
        "rmse": float(
            np.sqrt(
                np.mean(
                    (prediction - data) ** 2
                )
            )
        ),
    }


## 6. Acquisition summaries

The first step is to check exactly what is being compared. The benchmark below records total measurements, diffusion-weighted measurements, unique directions, and b-value shells. This lets the final ranking be expressed not only as "best fit" but also in terms of acquisition size.

In [ ]:
acquisition_summary_rows = []
for name, acq in ACQUISITIONS.items():
    bvals = np.asarray(acq.b_values, dtype=float)
    nonzero = bvals > 1e-8
    unique_b = np.unique(np.round(bvals[nonzero], 6))

    acquisition_summary_rows.append({
        "acquisition": name,
        "n_measurements": int(acq.n_measurements),
        "n_dw": int(nonzero.sum()),
        "n_b0": int((~nonzero).sum()),
        "n_b_shells": int(unique_b.size + int((~nonzero).any())),
        "n_unique_directions": (
            int(np.unique(np.round(acq.gradients[nonzero], 6), axis=0).shape[0])
            if acq.gradients is not None and nonzero.any()
            else 0
        ),
    })

acquisition_summary = pd.DataFrame(
    acquisition_summary_rows
).sort_values("n_measurements")

display(acquisition_summary)

for name, acq in ACQUISITIONS.items():
    print(f"\n{name}")
    acq.print_summary()


## 7. Sanity check before the full benchmark

Run one truth/noise realisation through every acquisition before committing to the larger repeated experiment.

In [ ]:
sanity_rows = []
truth = TRUTH_BY_SCENARIO["truth_2"]

for acquisition_name, acquisition in ACQUISITIONS.items():
    truth_signal = simulate_signal(
        tort2_model(
            theta=truth["theta"],
            phi=truth["phi"],
        ),
        acquisition,
        truth,
    )
    data = add_gaussian_noise(
        truth_signal,
        SNR,
        BASE_SEED,
    )

    noddi = make_noddi_estimate(
        truth,
        acquisition_name,
        NODDI_SEED,
    )

    try:
        fit = fit_tort2_ls(
            data,
            acquisition,
            acquisition_name,
            truth,
            noddi,
            seed=BASE_SEED,
            theta=ORIENTATION_BY_ACQUISITION[acquisition_name][0],
            phi=ORIENTATION_BY_ACQUISITION[acquisition_name][1],
        )
        fraction_sum = (
            fit["neurite_fraction"]
            + fit["extra_fraction"]
            + fit["soma_fraction"]
            + fit["csf_fraction"]
        )
        sanity_rows.append({
            "acquisition": acquisition_name,
            "success": fit["success"],
            "rmse": fit["rmse"],
            "r_soma": fit["r_soma"],
            "f_soma": fit["soma_fraction"],
            "f_fw": fit["csf_fraction"],
            "fraction_sum": fraction_sum,
        })
    except Exception as exc:
        sanity_rows.append({
            "acquisition": acquisition_name,
            "success": False,
            "error": repr(exc),
        })

display(pd.DataFrame(sanity_rows))


## 8. Main LS acquisition benchmark

For every acquisition × truth scenario × noise repeat:

- simulate the full Tort2 truth;
- add Gaussian noise at SNR=20;
- generate a NODDI estimate according to the configured NODDI performance;
- fit with Tort2 using moderate NODDI-informed Gaussian-style priors;
- checkpoint after every fit.

By default, NODDI estimates are generated **per repeat**, which is more realistic for an acquisition-comparison experiment. Set `NODDI_ESTIMATE_MODE = "fixed_per_truth"` if you want to reproduce the fixed-NODDI-estimate design used in the previous notebook. The previous notebook itself explicitly reused one NODDI estimate per truth scenario. fileciteturn8file0L21-L32

In [ ]:
results = []

for acquisition_name, acquisition in ACQUISITIONS.items():
    print(f"\n=== Acquisition: {acquisition_name} ===", flush=True)

    for scenario_name in TRUTH_SCENARIOS_TO_RUN:
        truth = TRUTH_BY_SCENARIO[scenario_name]
        truth_signal = simulate_signal(
            tort2_model(
                theta=truth["theta"],
                phi=truth["phi"],
            ),
            acquisition,
            truth,
        )

        fixed_noddi = None
        if NODDI_ESTIMATE_MODE == "fixed_per_truth":
            fixed_noddi = make_noddi_estimate(
                truth,
                acquisition_name,
                NODDI_SEED + list(TRUTH_SCENARIOS).index(scenario_name),
            )

        for rep in range(N_REPEATS):
            seed = BASE_SEED + rep
            data = add_gaussian_noise(
                truth_signal,
                SNR,
                seed,
            )

            noddi = fixed_noddi
            if noddi is None:
                noddi = make_noddi_estimate(
                    truth,
                    acquisition_name,
                    NODDI_SEED + 1000 * rep + sum((i + 1) * ord(ch) for i, ch in enumerate(acquisition_name)),
                )

            t0 = time.time()

            try:
                fit = fit_tort2_ls(
                    data,
                    acquisition,
                    acquisition_name,
                    truth,
                    noddi,
                    seed=seed,
                    theta=ORIENTATION_BY_ACQUISITION[acquisition_name][0],
                    phi=ORIENTATION_BY_ACQUISITION[acquisition_name][1],
                )

                row = {
                    "acquisition": acquisition_name,
                    "scenario": scenario_name,
                    "repeat": rep,
                    "seed": seed,
                    "success": fit["success"],
                    "rmse": fit["rmse"],
                    "n_evaluations": fit["n_evaluations"],
                    "runtime_s": time.time() - t0,
                    "true_r_soma": truth["r_soma"],
                    "true_soma_fraction": truth["soma_fraction"],
                    "true_f_fw": truth["csf_fraction"],
                }
                for parameter in [
                    "S0",
                    "r_soma",
                    "kappa",
                    "neurite_fraction",
                    "extra_fraction",
                    "soma_fraction",
                    "csf_fraction",
                    "d_in",
                    "d_soma",
                    "d_csf",
                ]:
                    if parameter in fit:
                        row[parameter] = fit[parameter]

                row["fraction_sum"] = (
                    row["neurite_fraction"]
                    + row["extra_fraction"]
                    + row["soma_fraction"]
                    + row["csf_fraction"]
                )
                row["r_soma_lower_bound"] = np.isclose(row["r_soma"], 1.0)
                row["r_soma_upper_bound"] = np.isclose(row["r_soma"], 15.0)

            except Exception as exc:
                row = {
                    "acquisition": acquisition_name,
                    "scenario": scenario_name,
                    "repeat": rep,
                    "seed": seed,
                    "success": False,
                    "rmse": np.nan,
                    "n_evaluations": np.nan,
                    "runtime_s": time.time() - t0,
                    "error": repr(exc),
                    "true_r_soma": truth["r_soma"],
                    "true_soma_fraction": truth["soma_fraction"],
                    "true_f_fw": truth["csf_fraction"],
                }
                print(f"  FAILED {scenario_name} rep {rep}: {exc}", flush=True)

            results.append(row)
            pd.DataFrame(results).to_csv(LS_RESULTS_FILE, index=False)

results_df = pd.DataFrame(results)
print("Rows:", len(results_df))
display(results_df.head())


In [ ]:
# =============================================================================
# 9. LS RECOVERY METRICS + ACQUISITION RANKING
# =============================================================================

PRIMARY_PARAMS = {
    "r_soma": "true_r_soma",
    "soma_fraction": "true_soma_fraction",
    "csf_fraction": "true_f_fw",
}

metric_rows = []

for (acquisition_name, scenario_name), sub in results_df.groupby(
    ["acquisition", "scenario"]
):
    for parameter, truth_column in PRIMARY_PARAMS.items():
        if parameter not in sub.columns:
            continue

        values = pd.to_numeric(sub[parameter], errors="coerce")
        truth_value = float(sub[truth_column].iloc[0])
        valid = values.notna()
        if valid.sum() == 0:
            continue

        errors = values[valid] - truth_value
        rmse = float(np.sqrt(np.mean(errors ** 2)))

        metric_rows.append({
            "acquisition": acquisition_name,
            "scenario": scenario_name,
            "parameter": parameter,
            "truth": truth_value,
            "mean": values[valid].mean(),
            "bias": errors.mean(),
            "mae": errors.abs().mean(),
            "rmse": rmse,
            "sd": values[valid].std(ddof=1),
            "relative_rmse": rmse / abs(truth_value) if truth_value != 0 else np.nan,
            "n_valid": int(valid.sum()),
            "n_total": len(values),
        })

metrics_df = pd.DataFrame(metric_rows)

acq_summary = (
    metrics_df
    .groupby(["acquisition", "parameter"])
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_relative_rmse=("relative_rmse", "mean"),
        worst_relative_rmse=("relative_rmse", "max"),
        mean_bias=("bias", "mean"),
    )
    .reset_index()
)

relative_pivot = acq_summary.pivot_table(
    index="acquisition",
    columns="parameter",
    values="mean_relative_rmse",
)

for parameter in PRIMARY_PARAMS:
    if parameter not in relative_pivot.columns:
        relative_pivot[parameter] = np.nan

relative_pivot["mean_primary_relative_rmse"] = relative_pivot[
    list(PRIMARY_PARAMS.keys())
].mean(axis=1)

scenario_worst = (
    metrics_df
    .groupby(["acquisition", "scenario"])["relative_rmse"]
    .mean()
    .groupby("acquisition")
    .max()
    .rename("worst_scenario_primary_relative_rmse")
)

success = (
    results_df
    .groupby("acquisition")["success"]
    .mean()
    .rename("success_rate")
)

acquisition_ranking = (
    relative_pivot
    .join(scenario_worst)
    .join(success)
    .join(acquisition_summary.set_index("acquisition"))
    .sort_values("mean_primary_relative_rmse")
)

acquisition_ranking["passes_default_criterion"] = (
    (acquisition_ranking["mean_primary_relative_rmse"] <= MAX_MEAN_PRIMARY_REL_RMSE)
    & (acquisition_ranking["worst_scenario_primary_relative_rmse"] <= MAX_WORST_SCENARIO_PRIMARY_REL_RMSE)
    & (acquisition_ranking["success_rate"] >= MIN_SUCCESS_RATE)
)

display(acquisition_ranking)


In [ ]:
# =============================================================================
# 10. "MINIMUM ACQUISITION" SELECTION
# =============================================================================

passing = acquisition_ranking[
    acquisition_ranking["passes_default_criterion"]
].copy()

if passing.empty:
    print("No acquisition meets the current default thresholds.")
    print("Use acquisition_ranking to choose the best trade-off, or relax the thresholds.")
else:
    minimum = passing.sort_values(
        ["n_measurements", "mean_primary_relative_rmse"]
    ).iloc[0]

    print("Minimum acquisition meeting the current thresholds:")
    print(minimum[[
        "n_measurements",
        "n_dw",
        "mean_primary_relative_rmse",
        "worst_scenario_primary_relative_rmse",
        "success_rate",
    ]])

# Plot performance vs acquisition size.
plot_df = acquisition_ranking.sort_values("n_measurements")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    plot_df["n_measurements"],
    plot_df["mean_primary_relative_rmse"],
    marker="o",
)
ax.axhline(
    MAX_MEAN_PRIMARY_REL_RMSE,
    linestyle="--",
    linewidth=1.2,
    label="Mean-RRMSE threshold",
)
ax.set_xlabel("Total acquisition measurements")
ax.set_ylabel("Mean primary relative RMSE")
ax.set_title("Acquisition size vs parameter-recovery performance")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
filename = PLOT_DIR / "acquisition_size_vs_performance.png"
fig.savefig(filename, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {filename}")


## 11. Best 3 acquisitions for MCMC

MCMC is reserved for the three acquisitions with the lowest mean primary relative RMSE from the LS benchmark. The posterior is sampled for `truth_2` by default. This keeps the MCMC experiment focused while allowing comparison of posterior geometry across acquisitions.

In [ ]:
TOP3_ACQUISITIONS = (
    acquisition_ranking
    .sort_values("mean_primary_relative_rmse")
    .head(3)
    .index
    .tolist()
)

print("Top 3 acquisitions:")
for i, name in enumerate(TOP3_ACQUISITIONS, 1):
    print(f"{i}. {name}")


In [ ]:
# =============================================================================
# 12. MCMC TARGET
# =============================================================================

# The MCMC target uses the same Tort2 model, fixed diffusivities, fixed
# DTI-informed theta/phi, and the same moderate Gaussian priors used by LS.
# Unlike LS, the priors are treated as actual prior densities and the ALR
# Jacobian is included because the chain samples unconstrained ALR coordinates.

def gaussian_logpdf(x, mu, sigma):
    return -0.5 * ((x - mu) / sigma) ** 2 - np.log(
        sigma * np.sqrt(2.0 * np.pi)
    )


def mcmc_log_posterior(x, data, acquisition, acquisition_name, truth, noddi):
    # x = [S0, r_soma, kappa, z_n, z_e, z_fw]
    if not (0.05 <= x[0] <= 2.0):
        return -np.inf
    if not (1.0 <= x[1] <= 15.0):
        return -np.inf
    if not (0.0 <= x[2] <= 64.0):
        return -np.inf

    fractions = alr_to_fractions(x[3:])
    values = {
        "S0": float(x[0]),
        "r_soma": float(x[1]),
        "kappa": float(x[2]),
        "theta": truth["theta"],
        "phi": truth["phi"],
        **fractions,
        "d_in": FIXED_MODEL_DIFFUSIVITIES["d_in"],
        "d_soma": FIXED_MODEL_DIFFUSIVITIES["d_soma"],
        "d_csf": FIXED_MODEL_DIFFUSIVITIES["d_csf"],
    }

    prediction = simulate_signal(
        tort2_model(
            theta=truth["theta"],
            phi=truth["phi"],
        ),
        acquisition,
        values,
    )

    sigma_data = float(np.max(np.abs(data)) / SNR)
    log_like = -0.5 * np.sum(((prediction - data) / sigma_data) ** 2)

    mae = noddi_mae(acquisition_name)
    prior_sigma = {
        key: 2.0 * value
        for key, value in mae.items()
    }

    log_prior = 0.0
    for parameter in [
        "kappa",
        "neurite_fraction",
        "extra_fraction",
        "csf_fraction",
    ]:
        log_prior += gaussian_logpdf(
            values[parameter],
            noddi[parameter],
            prior_sigma[parameter],
        )

    # Add simplex-transform Jacobian.
    log_prior += alr_jacobian_log(fractions)

    return float(log_like + log_prior)


def vector_from_fit(fit):
    return np.array([
        fit["S0"],
        fit["r_soma"],
        fit["kappa"],
        *fractions_to_alr(
            fit["neurite_fraction"],
            fit["extra_fraction"],
            fit["soma_fraction"],
            fit["csf_fraction"],
        ),
    ])


def run_random_walk_mcmc(
    initial,
    log_target,
    *,
    steps,
    burn,
    thin,
    seed,
    proposal_scales,
):
    rng = np.random.default_rng(seed)
    current = np.asarray(initial, dtype=float).copy()
    current_lp = log_target(current)

    samples = []
    accepted = 0

    for step in range(steps):
        proposal = current + rng.normal(
            0.0,
            proposal_scales,
        )
        proposal_lp = log_target(proposal)

        if np.isfinite(proposal_lp):
            if np.log(rng.random()) < proposal_lp - current_lp:
                current = proposal
                current_lp = proposal_lp
                accepted += 1

        if step >= burn and ((step - burn) % thin == 0):
            samples.append(current.copy())

    return np.asarray(samples), accepted / float(steps)


In [ ]:
# =============================================================================
# 13. RUN MCMC FOR TOP 3 ACQUISITIONS
# =============================================================================

mcmc_outputs = {}

for i, acquisition_name in enumerate(TOP3_ACQUISITIONS):
    acquisition = ACQUISITIONS[acquisition_name]
    truth = TRUTH_BY_SCENARIO[MCMC_TRUTH_SCENARIO]

    truth_signal = simulate_signal(
        tort2_model(
            theta=truth["theta"],
            phi=truth["phi"],
        ),
        acquisition,
        truth,
    )

    # One representative dataset for posterior-geometry assessment.
    data = add_gaussian_noise(
        truth_signal,
        SNR,
        MCMC_SEED + i,
    )

    noddi = make_noddi_estimate(
        truth,
        acquisition_name,
        NODDI_SEED + i,
    )

    fit = fit_tort2_ls(
        data,
        acquisition,
        acquisition_name,
        truth,
        noddi,
        seed=MCMC_SEED + i,
    )

    x0 = vector_from_fit(fit)

    chains = []
    acceptances = []

    for chain in range(MCMC_CHAINS):
        chain_rng = np.random.default_rng(
            MCMC_SEED + 100 * i + chain
        )
        start = x0.copy()
        start += chain_rng.normal(
            0.0,
            MCMC_PROPOSAL_SCALES * 2.0,
        )

        samples, acceptance = run_random_walk_mcmc(
            start,
            lambda x: mcmc_log_posterior(
                x,
                data,
                acquisition,
                acquisition_name,
                truth,
                noddi,
            ),
            steps=MCMC_STEPS,
            burn=MCMC_BURN,
            thin=MCMC_THIN,
            seed=MCMC_SEED + 1000 * i + chain,
            proposal_scales=MCMC_PROPOSAL_SCALES,
        )

        chains.append(samples)
        acceptances.append(acceptance)

    samples = np.concatenate(chains, axis=0)

    # Convert posterior samples back to physical parameters.
    physical = []
    for sample in samples:
        fractions = alr_to_fractions(sample[3:])
        physical.append({
            "S0": sample[0],
            "r_soma": sample[1],
            "kappa": sample[2],
            **fractions,
        })

    posterior_df = pd.DataFrame(physical)
    posterior_df.to_csv(
        MCMC_RESULTS_DIR / f"{acquisition_name}_posterior.csv",
        index=False,
    )

    mcmc_outputs[acquisition_name] = {
        "posterior": posterior_df,
        "acceptance": acceptances,
        "ls_fit": fit,
        "noddi": noddi,
    }

    print(
        acquisition_name,
        "mean acceptance:",
        np.mean(acceptances),
        "samples:",
        len(posterior_df),
    )


In [ ]:
# =============================================================================
# 14. MCMC POSTERIOR SUMMARY + DEGENERACY CORRELATIONS
# =============================================================================

POSTERIOR_PARAMETERS = [
    "r_soma",
    "soma_fraction",
    "csf_fraction",
    "kappa",
    "neurite_fraction",
    "extra_fraction",
]

mcmc_summary_rows = []

for acquisition_name, output in mcmc_outputs.items():
    posterior = output["posterior"]

    for parameter in POSTERIOR_PARAMETERS:
        values = posterior[parameter].to_numpy()
        q025, q50, q975 = np.quantile(
            values,
            [0.025, 0.50, 0.975],
        )

        mcmc_summary_rows.append({
            "acquisition": acquisition_name,
            "parameter": parameter,
            "mean": values.mean(),
            "sd": values.std(ddof=1),
            "q025": q025,
            "median": q50,
            "q975": q975,
        })

mcmc_summary_df = pd.DataFrame(
    mcmc_summary_rows
)
display(mcmc_summary_df)


# Maximum absolute posterior correlation among primary/nuisance parameters.
correlation_rows = []
for acquisition_name, output in mcmc_outputs.items():
    posterior = output["posterior"][POSTERIOR_PARAMETERS]
    corr = posterior.corr()
    corr_values = corr.to_numpy().copy()
    np.fill_diagonal(corr_values, np.nan)
    max_idx = np.unravel_index(
        np.nanargmax(np.abs(corr_values)),
        corr_values.shape,
    )

    correlation_rows.append({
        "acquisition": acquisition_name,
        "max_abs_correlation": abs(corr_values[max_idx]),
        "parameter_1": POSTERIOR_PARAMETERS[max_idx[0]],
        "parameter_2": POSTERIOR_PARAMETERS[max_idx[1]],
    })

correlation_df = pd.DataFrame(correlation_rows)
display(correlation_df)


In [ ]:
# Pairwise posterior plots for the top 3 acquisitions.
PAIR_PARAMETERS = [
    "r_soma",
    "soma_fraction",
    "csf_fraction",
    "kappa",
]

for acquisition_name, output in mcmc_outputs.items():
    posterior = output["posterior"]

    fig, axes = plt.subplots(
        len(PAIR_PARAMETERS),
        len(PAIR_PARAMETERS),
        figsize=(12, 12),
        constrained_layout=True,
    )

    for i, p1 in enumerate(PAIR_PARAMETERS):
        for j, p2 in enumerate(PAIR_PARAMETERS):
            ax = axes[i, j]

            if i == j:
                ax.hist(
                    posterior[p1],
                    bins=35,
                )
            elif i > j:
                ax.scatter(
                    posterior[p2],
                    posterior[p1],
                    s=2,
                    alpha=0.15,
                )
            else:
                ax.axis("off")
                continue

            if i == len(PAIR_PARAMETERS) - 1:
                ax.set_xlabel(p2)
            if j == 0:
                ax.set_ylabel(p1)
            ax.grid(alpha=0.2)

    fig.suptitle(
        f"Posterior geometry: {acquisition_name}",
        y=1.02,
    )

    filename = (
        MCMC_RESULTS_DIR
        / f"{acquisition_name}_posterior_pairs.png"
    )
    fig.savefig(
        filename,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(fig)
    print(f"Saved: {filename}")


## 15. Fixed-diffusivity robustness

The fit remains fixed to the nominal values:

- `d_in = 2.2`
- `d_soma = 3.0`
- `d_fw/d_csf = 3.0`

while the **synthetic truth** is deliberately moved away from the fixed value. This asks:

> How much bias do we introduce into the parameters of interest if the assumed fixed diffusivity is wrong?

The default mode changes one true diffusivity at a time. `full_grid` is available but is substantially more expensive.

In [ ]:
# =============================================================================
# 16. ROBUSTNESS BENCHMARK ON BEST 3 ACQUISITIONS
# =============================================================================

if ROBUSTNESS_MODE == "one_at_a_time":
    robustness_scenarios = []
    for parameter, factors in MISFIX_FACTORS.items():
        for factor in factors:
            robustness_scenarios.append({
                "misfixed_parameter": parameter,
                "factor": float(factor),
                "d_in": TRUE_BASE["d_in"],
                "d_soma": TRUE_BASE["d_soma"],
                "d_csf": TRUE_BASE["d_csf"],
            })
            robustness_scenarios[-1][parameter] *= factor
else:
    robustness_scenarios = []
    for fi, fs, fc in itertools.product(
        MISFIX_FACTORS["d_in"],
        MISFIX_FACTORS["d_soma"],
        MISFIX_FACTORS["d_csf"],
    ):
        robustness_scenarios.append({
            "misfixed_parameter": "joint",
            "factor_d_in": float(fi),
            "factor_d_soma": float(fs),
            "factor_d_csf": float(fc),
            "d_in": TRUE_BASE["d_in"] * fi,
            "d_soma": TRUE_BASE["d_soma"] * fs,
            "d_csf": TRUE_BASE["d_csf"] * fc,
        })

robust_results = []

for acquisition_name in TOP3_ACQUISITIONS:
    acquisition = ACQUISITIONS[acquisition_name]

    for scenario_name in TRUTH_SCENARIOS_TO_RUN:
        baseline_truth = TRUTH_BY_SCENARIO[scenario_name]

        for misfix_id, misfix in enumerate(robustness_scenarios):
            truth = dict(baseline_truth)
            truth.update({
                "d_in": misfix["d_in"],
                "d_soma": misfix["d_soma"],
                "d_csf": misfix["d_csf"],
            })

            truth_signal = simulate_signal(
                tort2_model(
                    theta=truth["theta"],
                    phi=truth["phi"],
                ),
                acquisition,
                truth,
            )

            for rep in range(ROBUSTNESS_REPEATS):
                seed = 30000 + 1000 * misfix_id + rep
                data = add_gaussian_noise(
                    truth_signal,
                    SNR,
                    seed,
                )

                noddi = make_noddi_estimate(
                    truth,
                    acquisition_name,
                    NODDI_SEED + seed,
                )

                try:
                    fit = fit_tort2_ls(
                        data,
                        acquisition,
                        acquisition_name,
                        truth,
                        noddi,
                        seed=seed,
                        fixed_diffusivities=FIXED_MODEL_DIFFUSIVITIES,
                    )

                    row = {
                        "acquisition": acquisition_name,
                        "scenario": scenario_name,
                        "misfixed_parameter": misfix["misfixed_parameter"],
                        "factor": misfix.get("factor", np.nan),
                        "factor_d_in": misfix.get("factor_d_in", np.nan),
                        "factor_d_soma": misfix.get("factor_d_soma", np.nan),
                        "factor_d_csf": misfix.get("factor_d_csf", np.nan),
                        "true_d_in": truth["d_in"],
                        "true_d_soma": truth["d_soma"],
                        "true_d_csf": truth["d_csf"],
                        "repeat": rep,
                        "success": fit["success"],
                        "rmse": fit["rmse"],
                        "r_soma": fit["r_soma"],
                        "soma_fraction": fit["soma_fraction"],
                        "csf_fraction": fit["csf_fraction"],
                        "true_r_soma": truth["r_soma"],
                        "true_soma_fraction": truth["soma_fraction"],
                        "true_f_fw": truth["csf_fraction"],
                    }
                except Exception as exc:
                    row = {
                        "acquisition": acquisition_name,
                        "scenario": scenario_name,
                        "misfixed_parameter": misfix["misfixed_parameter"],
                        "factor": misfix.get("factor", np.nan),
                        "factor_d_in": misfix.get("factor_d_in", np.nan),
                        "factor_d_soma": misfix.get("factor_d_soma", np.nan),
                        "factor_d_csf": misfix.get("factor_d_csf", np.nan),
                        "true_d_in": truth["d_in"],
                        "true_d_soma": truth["d_soma"],
                        "true_d_csf": truth["d_csf"],
                        "repeat": rep,
                        "success": False,
                        "rmse": np.nan,
                        "error": repr(exc),
                        "true_r_soma": truth["r_soma"],
                        "true_soma_fraction": truth["soma_fraction"],
                        "true_f_fw": truth["csf_fraction"],
                    }

                robust_results.append(row)
                pd.DataFrame(robust_results).to_csv(
                    ROBUSTNESS_RESULTS_FILE,
                    index=False,
                )

robustness_df = pd.DataFrame(robust_results)
print("Rows:", len(robustness_df))
display(robustness_df.head())


In [ ]:
# =============================================================================
# 17. ROBUSTNESS SUMMARY
# =============================================================================

robust_metric_rows = []

for keys, sub in robustness_df.groupby([
    "acquisition",
    "scenario",
    "misfixed_parameter",
    "factor",
]):
    acquisition_name, scenario_name, misfixed_parameter, factor = keys

    truth_r = float(sub["true_r_soma"].iloc[0])
    truth_s = float(sub["true_soma_fraction"].iloc[0])
    truth_fw = float(sub["true_f_fw"].iloc[0])

    for parameter, truth_value in [
        ("r_soma", truth_r),
        ("soma_fraction", truth_s),
        ("csf_fraction", truth_fw),
    ]:
        values = pd.to_numeric(sub[parameter], errors="coerce")
        valid = values.notna()
        if valid.sum() == 0:
            continue
        errors = values[valid] - truth_value
        robust_metric_rows.append({
            "acquisition": acquisition_name,
            "scenario": scenario_name,
            "misfixed_parameter": misfixed_parameter,
            "factor": factor,
            "parameter": parameter,
            "bias": errors.mean(),
            "mae": errors.abs().mean(),
            "rmse": np.sqrt(np.mean(errors ** 2)),
            "relative_rmse": (
                np.sqrt(np.mean(errors ** 2)) / abs(truth_value)
                if truth_value != 0 else np.nan
            ),
            "n_valid": int(valid.sum()),
        })

robust_metrics = pd.DataFrame(robust_metric_rows)
display(robust_metrics.sort_values([
    "acquisition",
    "misfixed_parameter",
    "factor",
    "parameter",
]))


In [ ]:
# Plot absolute bias in the primary biological parameters as each fixed diffusivity
# is deliberately misspecified.

for acquisition_name in TOP3_ACQUISITIONS:
    for misfixed_parameter in ["d_in", "d_soma", "d_csf"]:

        plot_data = robust_metrics[
            (robust_metrics["acquisition"] == acquisition_name)
            & (robust_metrics["misfixed_parameter"] == misfixed_parameter)
            & (robust_metrics["parameter"].isin(["r_soma", "soma_fraction", "csf_fraction"]))
        ].copy()

        if plot_data.empty:
            continue

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(15, 4.5),
            constrained_layout=True,
        )

        for ax, parameter in zip(
            axes,
            ["r_soma", "soma_fraction", "csf_fraction"],
        ):
            sub = plot_data[plot_data["parameter"] == parameter]
            if sub.empty:
                ax.axis("off")
                continue

            x = sub["factor"]
            y = sub["bias"]
            ax.plot(x, y, marker="o")
            ax.axhline(0.0, linestyle="--", linewidth=1.0)
            ax.set_xlabel(f"True / fixed {misfixed_parameter}")
            ax.set_ylabel(f"Bias in {parameter}")
            ax.set_title(parameter)
            ax.grid(alpha=0.25)

        fig.suptitle(
            f"Fixed-diffusivity robustness: {acquisition_name} — {misfixed_parameter}",
            y=1.02,
        )

        filename = (
            PLOT_DIR
            / f"robustness_{acquisition_name}_{misfixed_parameter}.png"
        )
        fig.savefig(
            filename,
            dpi=180,
            bbox_inches="tight",
        )
        plt.close(fig)
        print(f"Saved: {filename}")


## Interpretation guide

### Minimum acquisition

The LS ranking should be interpreted in terms of:

- accuracy of `r_soma`, `f_soma`, and `f_fw`;
- worst-case truth scenario, not only the average;
- successful fit rate;
- acquisition size / number of measurements;
- signal RMSE.

The "minimum" acquisition is therefore the **smallest acquisition that passes the user-defined recovery thresholds**, not simply the acquisition with the smallest RMSE.

### MCMC / degeneracy

For the best three acquisitions, inspect:

- posterior width of `r_soma` and `f_soma`;
- whether the posterior is elongated or multimodal;
- strong correlations such as `r_soma–f_soma`, `r_soma–kappa`, or `r_soma–f_fw`;
- whether different acquisitions produce substantially different posterior geometry.

A low LS RMSE does not by itself establish identifiability. The MCMC stage is specifically intended to determine whether similarly good signal fits correspond to broad or highly correlated parameter posteriors.

### Fixed diffusivity robustness

A fit can look excellent while relying on an incorrect fixed `d_in`, `d_soma`, or `d_fw`. The robustness analysis asks how much bias that introduces into the parameters of interest. In particular, look for systematic changes in `r_soma` and `f_soma` when the true diffusivity differs from the assumed fixed value.
